Q1

(a) None 
(b) '2026-05-06' 
(c) ['2026-05-06', '2026-05-18'] 
(d) [('2026', '05', '06'), ('2026', '05', '18')] 
(e) ['2026-05-06', '2026-05-18']   

설명: (c)는 캡처 그룹이 없어 일치하는 전체 문자열을 반환하고, (d)는 캡처 그룹 ()을 사용하여 추출된 텍스트들을 튜플로 묶어 반환한다. 반면 (e)는 비캡처 그룹 (?:)을 사용하여 패턴을 논리적으로 묶기만 하고 별도로 캡처하지 않으므로 (c)와 동일하게 전체 매칭 문자열을 돌려준다. 

In [ ]:
Q2

(a) '[T]!' 
(b) '[T]안녕[T] [T]세상[T]!' 
(c) '[T]안녕[T] [T]세상[T]!' 
(d) '수강생<30>명, 조교<3>명' 
(e) '수강생<\x01>명, 조교<\x01>명'

설명 (i): (a)는 탐욕적(Greedy) 수량자인 .+를 사용하여 첫 <부터 문장 끝의 가장 마지막 >까지 한 번에 길게 매칭한 반면
         (b)는 게으른(Lazy) 수량자인 .+?를 사용하여 < 이후 처음 만나는 >까지만 짧게 끊어서 매칭했기 때문
설명 (ii): (d)는 원시 문자열(r"...")을 사용하여 \1이 정규식의 1번 캡처 그룹(추출된 숫자)으로 올바르게 해석되지만
         (e)는 일반 문자열이므로 파이썬이 \1을 아스키코드 1번(SOH, 제어 문자)으로 자체 해석하여 이스케이프 해버렸기 때문

Q3-1 코드

In [3]:
import re
from collections import Counter

# 동일한 패턴이 반복 사용되므로 미리 컴파일 (성능 최적화)
pat_url = re.compile(r"https?://\S+")
pat_html = re.compile(r"<[^>]+>")
pat_email = re.compile(r"[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}")
pat_phone = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
pat_mention = re.compile(r"@\w+")
pat_hashtag = re.compile(r"#\w+")
pat_kr_con_vow = re.compile(r"[ㄱ-ㅣ]+")
pat_ws = re.compile(r"\s+")

# 해시태그 추출용 (괄호로 캡처 그룹 지정)
pat_hashtag_ext = re.compile(r"#(\w+)")

def clean_post(post: str) -> str:
    # 6단계 정제 파이프라인
    text = pat_url.sub(" ", post)
    text = pat_html.sub("", text)
    text = pat_email.sub("[이메일]", text)
    text = pat_phone.sub("[전화]", text)
    text = pat_mention.sub(" ", text)
    text = pat_hashtag.sub(" ", text)
    text = pat_kr_con_vow.sub("", text)
    text = pat_ws.sub(" ", text).strip()
    return text

def extract_hashtags(post: str) -> list[str]:
    # 캡처 그룹(\w+)을 사용했으므로 #을 제외한 알맹이만 리스트로 반환됨
    return pat_hashtag_ext.findall(post)

def analyze_posts(posts: list[str]) -> dict:
    total_masked = 0
    all_hashtags = []
    cleaned_lengths = []

    for post in posts:
        # 해시태그 추출 (원본에서)
        all_hashtags.extend(extract_hashtags(post))

        # 마스킹 횟수 계산을 위해 1, 2단계만 적용한 임시 텍스트 생성
        temp = pat_url.sub(" ", post)
        temp = pat_html.sub("", temp)
        
        # re.subn을 활용하여 마스킹된 치환 횟수 누적
        temp, email_cnt = pat_email.subn("[이메일]", temp)
        temp, phone_cnt = pat_phone.subn("[전화]", temp)
        total_masked += (email_cnt + phone_cnt)

        # 정제 및 글자 수 계산
        cleaned = clean_post(post)
        cleaned_lengths.append(len(cleaned))

    # 평균 길이 계산 (소수점 2자리 반올림)
    avg_len = round(sum(cleaned_lengths) / len(posts), 2) if posts else 0.0

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": avg_len,
        "hashtag_counts": dict(Counter(all_hashtags).most_common()),
        "masked_count": total_masked
    }

# ---------------- 테스트 실행부 ----------------
posts = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ",
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠㅠㅠㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강."
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    "여러 공백과\n\n\n줄바꿈이 많은 텍스트",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr"
]

print("(1) 각 게시물에 대한 clean_post 반환값:")
for p in posts:
    print(repr(clean_post(p)))

print("\n(2) analyze_posts 반환 딕셔너리:")
print(analyze_posts(posts))

(1) 각 게시물에 대한 clean_post 반환값:
'오늘 수업 진짜 재밌었음!! 감사'
'자료:'
'팀플 어디서 모이지 카톡'
'중요: 다음 시험 범위는 1-15강.문의는 [이메일] ([전화])로!'
'여러 공백과 줄바꿈이 많은 텍스트'
'진짜 좋다'

(2) analyze_posts 반환 딕셔너리:
{'posts_n': 6, 'avg_length_after_clean': 15.83, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


Q3-2 실행 결과

(1) 각 게시물에 대한 clean_post 반환값:
'오늘 수업 진짜 재밌었음!! 감사'
'자료:'
'팀플 어디서 모이지 카톡'
'중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!'
'여러 공백과 줄바꿈이 많은 텍스트'
'진짜 좋다'

(2) analyze_posts 반환 딕셔너리:
{'posts_n': 6, 'avg_length_after_clean': 15.83, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}

Q3-3 설명

이메일 마스킹(단계 3)을 멘션 제거(단계 4)보다 반드시 먼저 수행해야 한다. 만약 순서를 바꾸어 단계 4를 먼저 실행하면 이메일 주소(예: mam3b@snu.ac.kr) 내부의 @snu 부분이 멘션 패턴(@\w+)으로 잘못 인식되어 통째로 날아가 버리고 결국 이메일을 정상적으로 마스킹하지 못해 데이터 오염이 발생한다.